In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
import numpy as np
import pandas as pd
import re
from bs4 import BeautifulSoup

/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [3]:
df = pd.read_excel('리스크이진분류.xlsx')
df.head(2)

,Unnamed: 0,naver_article_id,writer_nickname,released_at,view_count,title,like_count,comment_count,comments,content,...,log_like_count,log_comment_count,text,is_cartier,is_product,deleted_comment_count,deleted_comment_ratio,label,pred_label,pred_prob
0,0,5536787,2024Y,2025-08-19 23:38:00,345,불가리 뱀이냐 까르띠에 팬더냐 🤣 뭐가 더 이쁜가요?,0,4,앗 전 압도적으로 2번이요|불가리 착용해보심 실물 넘예뻐요!!\n링부분도 투보가스라...,"\n <div class=""se-component se-...",...,0.0,1.609438,불가리 뱀이냐 까르띠에 팬더냐 🤣 뭐가 더 이쁜가요?\n무기명 투표 중 반지 투보가...,1,1,0,0.0,0,0,0.127695


In [4]:
df.isnull().sum()

Unnamed: 0                 0
naver_article_id           0
writer_nickname            0
released_at                0
view_count                 0
title                      0
like_count                 0
comment_count              0
comments                 241
content                    0
clean_content              5
log_view_count             0
log_like_count             0
log_comment_count          0
text                       0
is_cartier                 0
is_product                 0
deleted_comment_count      0
deleted_comment_ratio      0
label                      0
pred_label                 0
pred_prob                  0
dtype: int64

In [ ]:
def extract_se_text(html):
    soup = BeautifulSoup(html, 'html.parser')
    se_text = soup.find(class_='se-text')
    if se_text:
        text = se_text.get_text(separator=' ').strip()
        text = text.replace('\u200b', '').replace('\xa0', '')
        return text
    else:
        return ''

df["se_text"] = df["content"].apply(extract_se_text)
df["text"] = df["title"].fillna('') + ' ' + df["se_text"].fillna('')
df["text"] = df["text"].str.strip()

In [4]:
df.isnull().sum()

Unnamed: 0                 0
naver_article_id           0
writer_nickname            0
released_at                0
view_count                 0
title                      0
like_count                 0
comment_count              0
comments                 241
content                    0
clean_content              5
log_view_count             0
log_like_count             0
log_comment_count          0
text                       0
is_cartier                 0
is_product                 0
deleted_comment_count      0
deleted_comment_ratio      0
label                      0
pred_label                 0
pred_prob                  0
dtype: int64

In [7]:
df_risk = df[df['pred_label'] == 1]
print(f"리스크 게시글 개수 : {len(df_risk)}개")

리스크 게시글 개수 : 1860개


In [ ]:
# 1. 키워드 사전 정의
risk_keywords = {
    "품질리스크": [
        # 외관 손상
        "스크래치", "기스", "흠집", "파손", "깨짐", "휘어짐",

        # 변색/광택
        "변색", "옐골화", "옐로우골드", "광", "광택",
        "기스광", "쇳덩이", "쇳덩어리", "묻어나", "이염",

        # 도금
        "도금", "벗겨",

        # 시계 기능
        "멈", "느려진", "오작동", "작동 안",

        # 부품
        "체인", "고장",

        # 가품
        "가품", "짝퉁", "짭", "정품",

        # 수리/보수
        "폴리싱", "오버홀",

        # 기타
        "불량", "하자", "이물질", "냄새", "품질",
        "노다이아", "민자",
    ],

    "운영리스크": [
        # 웨이팅/오픈런
        "웨이팅", "대기", "오픈런", "까픈런",

        # 은어
        "까르띠에 대전", "존버",

        # 공홈/온라인
        "공홈", "공식홈페이지", "온라인", "앱", "사이트",
        "새로고침", "새로 고침", "품절",

        # 배송
        "배송", "지연", "오배송", "늦",

        # 입고/재고
        "입고", "재고",

        # AS/수선
        "AS", "수선", "폴리싱", "오버홀", "주문",
    ],

    "평판리스크": [
        # 감정/반응
        "실망", "최악", "별로", "화남", "역대급", "황당",
        "기분나쁨", "기분 상하", "감정 상하",

        # 브랜드 이슈
        "논란", "불매", "블랙리스트",

        # 직원 태도
        "불친절", "직원", "셀러", "매장 직원", "응대", "태도", "갑질", "무시",
        "진상", "싸가지", "경우가 없", "기본이 없", "기본도 없", "기본도 안",
        "일하기 싫", "성의없", "성의 없", "건성", "쌩까",
        "쳐다도 안", "눈도 안", "대꾸도 안", "무반응",
        "코빼기도", "나몰라라",
        "직원 교육", "교육이 안", "교육을 안",

        # 컴플레인 예고
        "컴플레인", "항의", "민원", "본사에", "신고",
        "진상 부려", "한 소리",

        # 희소성/브랜드 가치 하락
        "흔하다", "흔해졌", "길거리", "아무나", "다 들고",
        "명품 아닌", "돈값",

        # 중고/리셀 가품
        "구구스", "중고", "리셀", "크림",
    ],

    "정책리스크": [
        # 가격 인상
        "가격인상", "가격 인상", "인상폭", "인상 폭",
        "또 올랐", "또 인상", "자꾸 올려", "계속 올려",

        # 환불/교환
        "환불", "교환거절", "환불불가",

        # 구매 제한
        "구매 제한", "1인1개",

        # 정책 일반
        "정책", "규정", "약관",

        # 고객 과실 전가
        "고객 책임", "고객책임", "과실", "소비자 과실",
        "책임 전가", "본인 과실",

        # 불량 판정 프로세스
        "본사", "감정", "불량 아니", "하자 아니",
        "프랑스", "판정", "인정 안", "책임 안",

        # 수리비
        "수리비", "무상수리", "유상",

        # 누락/미제공
        "보증서", "파우치", "정품 등록", "등록 놓쳐",
    ],
}

# 2. 분류 함수
# 우선순위 정의 (비즈니스 판단으로 조정 가능)
priority = ["품질리스크", "정책리스크", "운영리스크", "평판리스크"]

def classify_risk(text):
    scores = {category: 0 for category in risk_keywords}

    for category, keywords in risk_keywords.items():
        for keyword in keywords:
            if keyword in text:
                scores[category] += 1

    max_score = max(scores.values())

    if max_score == 0:
        return "미분류"

    # 동점이면 우선순위로 결정
    top_categories = [c for c, s in scores.items() if s == max_score]

    for p in priority:
        if p in top_categories:
            return p

df_risk["risk_category"] = df_risk["text"].apply(classify_risk)

In [9]:
# 1. 분류 결과 비율 확인
print(df_risk["risk_category"].value_counts())

# 2. 미분류 비율 확인
no_categorize = (df_risk["risk_category"] == "미분류").sum() / len(df_risk) * 100
print(f"미분류 비율: {no_categorize:.1f}%")

risk_category
미분류      955
운영리스크    491
품질리스크    175
평판리스크    158
정책리스크     81
Name: count, dtype: int64
미분류 비율: 51.3%


In [10]:
pd.set_option('display.max_colwidth', None)
# 3. 미분류 샘플 확인 (BERT fallback 전에 육안으로 먼저 봐)
df_risk[df_risk["risk_category"] == "미분류"]["text"].sample(50)

In [ ]:
df_risk[df_risk["risk_category"] == "미분류"]["text"].sample(50).to_excel("미분류50개.xlsx")

In [13]:
! pip install sentence_transformers
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1. 한국어 사전학습 모델 로드
model = SentenceTransformer('snunlp/KR-SBERT-V40K-klueNLI-augSTS')

# 2. 카테고리 대표 문장 정의
category_descriptions = {
    "품질리스크": "도금이 벗겨지고 기스가 나고 변색되고 옐골화가 되고 광이 죽고 쇳덩이처럼 변하고 가품이 의심되고 시계가 느려지거나 멈추고 부품이 고장난다",
    "운영리스크": "웨이팅이 길고 까픈런이 힘들고 재고가 없고 배송이 지연되고 AS가 안 된다",
    "평판리스크": "직원이 불친절하고 싸가지 없고 응대가 최악이고 브랜드에 실망하고 불매한다",
    "정책리스크": "가격을 자꾸 인상하고 환불이 거절되고 수리비를 청구하고 구매를 제한한다"
}

# 3. 카테고리 임베딩 미리 계산
category_embeddings = {
    cat: model.encode(desc)
    for cat, desc in category_descriptions.items()
}

# 4. fallback 함수
def classify_unclassified(text):
    text_embedding = model.encode(text)

    similarities = {
        cat: cosine_similarity([text_embedding], [emb])[0][0]
        for cat, emb in category_embeddings.items()
    }

    return max(similarities, key=similarities.get)

# 5. 미분류만 fallback 적용
mask = df_risk["risk_category"] == "미분류"
df_risk.loc[mask, "risk_category"] = df_risk.loc[mask, "text"].apply(classify_unclassified)

# 6. 결과 확인
print(df_risk["risk_category"].value_counts())

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/467M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

risk_category
운영리스크    1181
품질리스크     363
평판리스크     165
정책리스크     151
Name: count, dtype: int64


In [14]:
df_risk[df_risk["risk_category"] == "운영리스크"]["text"].sample(20)

In [15]:
df_quality = df_risk[df_risk["risk_category"] == "품질리스크"]
df_operation = df_risk[df_risk["risk_category"] == "운영리스크"]
df_reputation = df_risk[df_risk["risk_category"] == "평판리스크"]
df_policy = df_risk[df_risk["risk_category"] == "정책리스크"]

print(len(df_quality), len(df_operation), len(df_reputation), len(df_policy))

363 1181 165 151


In [15]:
# 각 버킷에서 키워드가 하나도 없는 글 제거
# 즉 rule-based에서 키워드로 잡힌 글만 남기기

def has_keyword(text, category):
    keywords = risk_keywords[category]
    return any(kw in text for kw in keywords)

df_quality_clean = df_quality[df_quality["text"].apply(
    lambda x: has_keyword(x, "품질리스크"))]

df_operation_clean = df_operation[df_operation["text"].apply(
    lambda x: has_keyword(x, "운영리스크"))]

df_reputation_clean = df_reputation[df_reputation["text"].apply(
    lambda x: has_keyword(x, "평판리스크"))]

df_policy_clean = df_policy[df_policy["text"].apply(
    lambda x: has_keyword(x, "정책리스크"))]

print(len(df_quality_clean), len(df_operation_clean),
      len(df_reputation_clean), len(df_policy_clean))

363 1181 165 151


In [63]:
!pip install bertopic -q
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer
import random
import numpy as np

embedding_model = SentenceTransformer('snunlp/KR-SBERT-V40K-klueNLI-augSTS')

stopwords = [
    # 자음/모음 단독 (ㄱㄱ, ㅠㅠ 등)
    'ㄱ', 'ㄴ', 'ㄷ', 'ㄹ', 'ㅁ', 'ㅂ', 'ㅅ', 'ㅇ', 'ㅈ', 'ㅊ', 'ㅋ', 'ㅌ', 'ㅍ', 'ㅎ',
    'ㄱㄱ', 'ㄱㄱㄱ', 'ㄱㄱㄱㄱ', 'ㅈㄴ', 'ㅈㄴㄱ',
    'ㅋㅋ', 'ㅋㅋㅋ', 'ㅎㅎ', 'ㅎㅎㅎ', 'ㅠㅠ', 'ㅠㅠㅠ', 'ㅜㅜ',

    # 까르띠에 브랜드명 (토픽 구분에 도움 안 됨)
    '까르띠에', '깔띠', '까르', '깔띠에',

    # 의미없는 부사/형용사
    '너무', '혹시', '하고', '제가', '저는', '같아요',
    '있어요', '싶어요', '했는데', '그냥', '많이',

    # 추가
    '정말', '진짜', '완전', '약간', '조금', '좀',
    '이제', '아직', '벌써', '드디어', '일단',
    '안녕하세요', '감사합니다', '감사해요', '부탁드려요',
    '궁금해요', '궁금합니다', '여쭤봐요', '올려요',
    '스몰', '미디움', '오리지널', '라지',  # 사이즈 표현

    # 추가2
    '더', '잘', '넘', '안', '것', '또', '글',
    '이', '가', '은', '는', '을', '를', '에', '의', '로', '으로',
    '하는', '되는', '있는', '같은', '이런', '그런',
    '정도', '경우', '때문', '부분',

    #제품명   
    '러브', '러브팔찌', '러브링', '러브브레이슬릿',
    '세르펜티', '클래쉬', '다무르', '앵끌루', '저스트앵끌루',
    '트리니티', '탱크', '탱머', '탱머스트', '산토스',
    '발롱블루', '발렉스', '팬더', '베누아',
    '팔찌', '반지', '목걸이', '시계', '귀걸이',  # 제품 카테고리
    '골드', '화골', '로골', '스틸', '가죽',  # 소재
    '다이아', '다이아몬드',

    # 기존 불용어에 추가
    '사이즈', '매장', '매장에서', '구매', '구입', '오늘',
    '분', '다들', '같이', '저도', '제',
    '1', '2', '3', '4', '5', '6', '7', '8', '15', '16', '17', '18' # 숫자/사이즈
    'ec',  # 오타/노이즈
    '떴어요', '근데', '다',

    # 불용어 추가
    '반클', '오닉스', '화이트골드', '프랑세즈', '풀파베',  # 제품명
    'ec', 'ㅠ', '있을까요', '했어요', '하나', '나을까요',  # 노이즈
]

vectorizer = CountVectorizer(
    stop_words=stopwords,
    token_pattern=r"(?u)\b\w+\b"  # 한글 토큰 인식
)

def run_bertopic(df_subset, category_name, nr_topics):
    docs = df_subset["text"].tolist()

    topic_model = BERTopic(
        embedding_model=embedding_model,
        vectorizer_model=vectorizer,
        nr_topics=nr_topics,
        min_topic_size=10,
        language="multilingual"
    )

    topics, probs = topic_model.fit_transform(docs)

    print(f"\n=== {category_name} ===")
    print(topic_model.get_topic_info())

    return topic_model, topics, probs

# 각 카테고리별 실행
model_quality, topics_q, probs_q = run_bertopic(df_quality_clean, "품질리스크", nr_topics=4)
model_operation, topics_o, probs_o = run_bertopic(df_operation_clean, "운영리스크", nr_topics=6)
model_reputation, topics_r, probs_r = run_bertopic(df_reputation_clean, "평판리스크", nr_topics=3)
model_policy, topics_p, probs_p = run_bertopic(df_policy_clean, "정책리스크", nr_topics=3)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: snunlp/KR-SBERT-V40K-klueNLI-augSTS
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



=== 품질리스크 ===
   Topic  Count             Name  \
0     -1     52    -1_옐골_여유_딱_vs   
1      0     78  0_기스_레이어드_계속_미니   
2      1     28  1_교환_다시_폴리싱_시계가   
3      2     17  2_화골이_변색_체인이_거의   

                                    Representation  \
0      [옐골, 여유, 딱, vs, 시크님들, 골라주세요, 지금, 뱅글, 미니, 링]   
1   [기스, 레이어드, 계속, 미니, 분들, 뱅글, 인상전, 이번에, 사진펑, 같아서]   
2     [교환, 다시, 폴리싱, 시계가, 바로, 제품, 보니, 1회, 와서, 네크리스]   
3  [화골이, 변색, 체인이, 거의, 5모티브, 팔찌는, 화이트, 됐는데, 남자, 스윗]   


=== 운영리스크 ===
   Topic  Count                Name  \
0     -1    190     -1_오픈런_인상_재고_공홈   
1      0     92    0_전_공홈에서_인상_클래쉬드   
2      1     85     1_대기_웨이팅_오픈런_평일   
3      2     50   2_스트랩_공홈_가죽줄_공홈에서   
4      3     45  3_공홈에서_인상_구매했어요_공홈   
5      4     29    4_공홈_주문_카톡이_주문완료   

                                     Representation  \
0  [오픈런, 인상, 재고, 공홈, 공홈에서, 웨이팅, 탱크머스트, 옐골, 없어서, 바로]   
1     [전, 공홈에서, 인상, 클래쉬드, 딱, 사이즈가, 하는데, 공홈, 미듐, 옐골]   
2        [대기, 웨이팅, 오픈런, 평일, 신강, 잠실, 신세계, 판교, 수, 지금]   
3  [스트랩, 공홈, 가죽줄, 공홈에서, 

In [64]:
model_quality.get_representative_docs(2)  

In [65]:
model_operation.get_representative_docs(2)

In [66]:
model_operation.get_representative_docs(0)

In [67]:
model_operation.get_representative_docs(3)

In [71]:
model_reputation.get_representative_docs(0)

In [69]:
model_policy.get_representative_docs(1)

In [72]:
print("품질리스크 전체:", len(df_quality))
print("품질리스크 clean:", len(df_quality_clean))
print("품질리스크 -1:", sum(1 for t in topics_q if t == -1))

print("운영리스크 전체:", len(df_operation))
print("운영리스크 clean:", len(df_operation_clean))
print("운영리스크 -1:", sum(1 for t in topics_o if t == -1))

print("평판리스크 전체:", len(df_reputation))
print("평판리스크 clean:", len(df_reputation_clean))
print("평판리스크 -1:", sum(1 for t in topics_r if t == -1))

print("정책리스크 전체:", len(df_policy))
print("정책리스크 clean:", len(df_policy_clean))
print("정책리스크 -1:", sum(1 for t in topics_p if t == -1))

품질리스크 전체: 363
품질리스크 clean: 175
품질리스크 -1: 52
운영리스크 전체: 1181
운영리스크 clean: 491
운영리스크 -1: 190
평판리스크 전체: 165
평판리스크 clean: 158
평판리스크 -1: 15
정책리스크 전체: 151
정책리스크 clean: 81
정책리스크 -1: 25


In [73]:
# 각 카테고리별 토픽 컬럼 추가
df_quality_clean["topic"] = topics_q
df_operation_clean["topic"] = topics_o
df_reputation_clean["topic"] = topics_r
df_policy_clean["topic"] = topics_p

# 토픽 이름 매핑
quality_topic_map = {
    -1: "노이즈",
    0: "기스우려_내구성불안",
    1: "외관불량_하자인정거부",
    2: "노이즈",
}

operation_topic_map = {
    -1: "노이즈",
    0: "노이즈",
    1: "웨이팅_오픈런",
    2: "재고부족_스트랩입고대기",
    3: "노이즈",
    4: "공홈재고부족",
}

reputation_topic_map = {
    -1: "노이즈",
    0: "노이즈",
    1: "셀러불친절_응대불량",
}

policy_topic_map = {
    -1: "노이즈",
    0: "교환환불정책불만",
    1: "가격인상불만",
}

# 매핑 적용
df_quality_clean["topic_name"] = df_quality_clean["topic"].map(quality_topic_map)
df_operation_clean["topic_name"] = df_operation_clean["topic"].map(operation_topic_map)
df_reputation_clean["topic_name"] = df_reputation_clean["topic"].map(reputation_topic_map)
df_policy_clean["topic_name"] = df_policy_clean["topic"].map(policy_topic_map)

# 전체 합치기
df_bertopic = pd.concat([
    df_quality_clean,
    df_operation_clean,
    df_reputation_clean,
    df_policy_clean
])

# 노이즈 제외
df_bertopic = df_bertopic[df_bertopic["topic_name"] != "노이즈"]

print(df_bertopic["topic_name"].value_counts())
print(f"\n최종 분석 데이터: {len(df_bertopic)}개")

topic_name
셀러불친절_응대불량      117
웨이팅_오픈런          85
기스우려_내구성불안       78
재고부족_스트랩입고대기     50
가격인상불만           32
공홈재고부족           29
외관불량_하자인정거부      28
교환환불정책불만         24
Name: count, dtype: int64

최종 분석 데이터: 443개


In [74]:
# 중복 텍스트 있는지 확인
print(df_risk["naver_article_id"].duplicated().sum())

0


In [75]:
# 1. rule-based + fallback 결과 (1860개, risk_category 컬럼 있음)
# 2. bertopic 결과 (443개, topic_name 컬럼 있음)

# bertopic 결과에서 필요한 컬럼만 추출
df_topic_result = df_bertopic[["naver_article_id", "topic_name"]]

# 원본 df에 topic_name 합치기
df_final = df_risk.merge(df_topic_result, on="naver_article_id", how="left")

# 확인
print(df_final.shape)
print(df_final["risk_category"].value_counts())
print(df_final["topic_name"].value_counts(dropna=True))

# 저장
df_final.to_excel("bertopic3.xlsx", index=False)

(1860, 25)
risk_category
운영리스크    1181
품질리스크     363
평판리스크     165
정책리스크     151
Name: count, dtype: int64
topic_name
셀러불친절_응대불량      117
웨이팅_오픈런          85
기스우려_내구성불안       78
재고부족_스트랩입고대기     50
가격인상불만           32
공홈재고부족           29
외관불량_하자인정거부      28
교환환불정책불만         24
Name: count, dtype: int64


In [76]:
# text가 결측인 게시글 확인
print(df_final[df_final['text'].isna()]['content'].head(3))

# 빈 문자열도 같이 확인
print((df_final['text'] == '').sum())  # 빈 문자열 개수
print(df_final['text'].isna().sum())   # 진짜 NaN 개수

Series([], Name: content, dtype: str)
0
0


In [77]:
df_final.isnull().sum()

Unnamed: 0                  0
naver_article_id            0
writer_nickname             0
released_at                 0
view_count                  0
title                       0
like_count                  0
comment_count               0
comments                   81
content                     0
clean_content               2
log_view_count              0
log_like_count              0
log_comment_count           0
text                        0
is_cartier                  0
is_product                  0
deleted_comment_count       0
deleted_comment_ratio       0
label                       0
pred_label                  0
pred_prob                   0
se_text                     0
risk_category               0
topic_name               1417
dtype: int64